# `priya_loader` — dark-matter particle positions & velocities

The IC bigfile (`ICS/<box>_<Ngrid>_99/`) is the **only** place particle positions
and velocities exist anywhere in the archive: MP-Gadget was configured to write
snapshots (`output/PART_*`), but the PRIYA archive kept only the Lyman-alpha
spectra — the snapshots themselves were never archived. So every plot below is
at the IC redshift, **z ≈ 99**; there is no lower-z particle snapshot to load
instead.

This notebook covers dark-matter particles (`ptype="dm"`) only, and does not
re-teach the simulation suite, the nine parameters, or the τ/flux side — see
[`quickstart.ipynb`](quickstart.ipynb) for that. It is a companion to
`quickstart.ipynb`'s §3 (IC density), zoomed in on the raw particle
positions/velocities rather than the meshed density.

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from priya_loader import discover_simulations, find_production_ic_dir, load_ic_particles

ROOT = os.environ.get(
    "PRIYA_ROOT",
    "/global/cfs/cdirs/desicollab/users/jibancat/priya/emu_full",
)
print("ROOT:", ROOT, "| exists:", Path(ROOT).is_dir())

sims = discover_simulations(ROOT)
print(len(sims), "simulations under ROOT")

# Not every simulation necessarily has its production-resolution IC staged yet
# (see quickstart.ipynb / HANDOFF.md on partial staging) -- take the first one
# that does.
chosen_sim, ic_dir = None, None
for sim in sims:
    d = find_production_ic_dir(sim.directory)
    if d is not None:
        chosen_sim, ic_dir = sim, d
        break
if ic_dir is None:
    raise RuntimeError(f"no simulation under {ROOT} has a staged production IC")
print("simulation:", chosen_sim.name)
print("ic_dir    :", ic_dir)

data, header = load_ic_particles(ic_dir, ptype="dm",
                                  columns=("Position", "Velocity"), subsample=64)
print(header)   # box, redshift, use_peculiar_velocity, velocity_units
print("particles loaded:", data["Position"].shape[0])

## Units: these velocities are already peculiar km/s

MP-GenIC can write the `Velocity` block in either of two conventions, and it
records which one was used in the IC header's `UsePeculiarVelocity` flag.
GenIC's declared default is `UsePeculiarVelocity = 1` (`genic/params.c:52`), and
PRIYA's `_genic_params.ini` does not override it — so **PRIYA's stored
`Velocity` is already peculiar km/s**.

Applying the Gadget-2 `sqrt(a)` rescaling on top of that anyway — the
conversion GenIC itself applies only when the flag is 0
(`libgenic/zeldovich.c:195-202`) — would be a **10x error at z≈99**
(`sqrt(a) = sqrt(1/100) ≈ 0.1`).

`load_ic_particles(..., velocity="peculiar_kms")` (the default used above) reads
the header flag and returns peculiar km/s regardless of which convention was
stored on disk; that is exactly `header["use_peculiar_velocity"]` /
`header["velocity_units"]` printed above. Pass `velocity="raw"` only if you want
the stored block untouched.

In [ ]:
# A thin slab in z picks out a spatial region to visualize. But `subsample=64`
# above is a STRIDED cut through GenIC's row-major on-disk order, so the
# surviving particles sit on widely-spaced Lagrangian z-planes (spacing ~
# subsample * box/ngrid) -- much thicker than this slab (box/nmesh_slab). So the
# slab cut alone does NOT reliably bound the arrow count: it can catch an entire
# ~ngrid^2-particle plane, or none. To keep the plot readable regardless, we
# additionally thin the REAL particles inside the slab down to a bounded number
# of arrows with a seeded RNG -- this only chooses which already-loaded
# particles to draw; it never synthesizes data. This is the Zel'dovich flow at
# z~99: particles stream from underdense into overdense regions, tracing the
# cosmic web before it has visibly formed in the density field.
box_kpc_h = header["box_kpc_h"]
nmesh_slab = 128                                        # slab thickness = box / nmesh_slab
slab = data["Position"][:, 2] < box_kpc_h / nmesh_slab
pos, vel = data["Position"][slab], data["Velocity"][slab]
print(f"{slab.sum()} particles in the z < box/{nmesh_slab} slab")

max_arrows = 500                                        # bound for a readable quiver plot
rng = np.random.default_rng(0)                          # seeded: reproducible, not synthetic data
if pos.shape[0] > max_arrows:
    keep = rng.choice(pos.shape[0], size=max_arrows, replace=False)
    pos, vel = pos[keep], vel[keep]
print(f"{pos.shape[0]} arrows plotted (randomly thinned from the slab for readability)")

plt.figure(figsize=(6.4, 6.4))
plt.scatter(pos[:, 0], pos[:, 1], s=3, c="0.3", alpha=0.6)
plt.quiver(pos[:, 0], pos[:, 1], vel[:, 0], vel[:, 1], color="C0", alpha=0.85)
plt.xlabel("x [comoving kpc/h]"); plt.ylabel("y [comoving kpc/h]")
plt.title(f"DM particles + peculiar velocity (thin slab), z = {header['redshift']:.1f}")
plt.gca().set_aspect("equal")
plt.tight_layout(); plt.show()

In [ ]:
# Linear (Zel'dovich) theory predicts the velocity scale from the *displacement*
# measured on these SAME particles, with no external normalisation:
#   v = a * H(a) * f(z) * Psi,   Psi = Position - q  (q = the Lagrangian grid site)
# so v_rms should land near a*H(a)*f(z)*sigma_disp. This mirrors
# tests/test_real_ic.py::test_real_ic_velocity_matches_linear_theory -- copying its
# unit handling exactly (sigma_disp needs BOTH /1000 and /hubble to go from comoving
# kpc/h to the comoving-Mpc convention units.hubble_z expects).
import bigfile

from priya_loader import units

with bigfile.File(str(ic_dir)) as bf:
    ngrid = int(round(bf["1/Position"].size ** (1.0 / 3.0)))   # from the full block, not the subsample
print("ngrid (production grid):", ngrid)

id_data, _ = load_ic_particles(ic_dir, ptype="dm", columns=("ID",), subsample=64)
ids = id_data["ID"].astype(np.int64)                      # same stride as cell above -> same particles
npart_total = ngrid ** 3
lag = (ids - 1) % npart_total                              # row-major Lagrangian index
q = np.stack([lag // (ngrid * ngrid), (lag // ngrid) % ngrid, lag % ngrid], axis=1) * (box_kpc_h / ngrid)
d = data["Position"] - q
d -= box_kpc_h * np.round(d / box_kpc_h)                    # periodic minimum image
sigma_disp_kpc_h = float(np.sqrt((d ** 2).sum(axis=1).mean()))

a, z = header["scale_factor"], header["redshift"]
om, ol, h = header["Omega0"], header["OmegaLambda"], header["hubble"]
H = units.hubble_z(z, om, ol, h)                            # km/s/Mpc (h already baked in)
f = units.growth_rate(z, om, ol)
v_linear = a * H * f * (sigma_disp_kpc_h / 1000.0 / h)       # kpc/h -> comoving Mpc, same as hubble_z's convention
print(f"sigma_disp = {sigma_disp_kpc_h:.2f} kpc/h  ->  linear-theory v = {v_linear:.3f} km/s")

vel = data["Velocity"]
fig, ax = plt.subplots(1, 4, figsize=(16, 3.3))
for i, comp in enumerate("xyz"):
    ax[i].hist(vel[:, i], bins=80, color="C0")
    ax[i].axvline(v_linear, color="k", ls="--", lw=1)
    ax[i].axvline(-v_linear, color="k", ls="--", lw=1)
    ax[i].set_title(f"v_{comp} [km/s]")
speed = np.sqrt((vel ** 2).sum(axis=1))
ax[3].hist(speed, bins=80, color="C1")
ax[3].axvline(v_linear, color="k", ls="--", lw=1, label=r"$aHf\sigma_{disp}$")
ax[3].set_title("|v| [km/s]"); ax[3].legend(fontsize=8)
plt.suptitle(f"DM peculiar velocity, z = {header['redshift']:.1f}")
plt.tight_layout(); plt.show()

print("Each component is ~Gaussian and zero-mean; |v| peaks near the a*H*f*sigma_disp")
print("marker. That agreement IS the units check -- a units bug (e.g. a stray or")
print("missing sqrt(a)) would be off by ~10x, not O(1).")

In [ ]:
# v is the potential flow of delta_1 in linear theory (v ~ i k/k^2 * delta_1), so a
# velocity-field slice and the matching delta_1 slice should visibly track each
# other -- infall onto overdensities -- even though these come from two different
# on-disk blocks (Velocity CIC-averaged per cell vs. ICDensity reshaped by ID).
from priya_loader import load_ic_density, load_ic_velocity_mesh

nmesh = 128   # divides both the 1536^3 and 3072^3 production grids
vf = load_ic_velocity_mesh(ic_dir, ptype="dm", nmesh=nmesh, field="velocity")
df = load_ic_density(ic_dir, ptype="dm", nmesh=nmesh, field="icdensity")
print("empty_cells (velocity mesh):", vf.meta["empty_cells"], "/", nmesh ** 3)

k = nmesh // 2
fig, ax = plt.subplots(1, 2, figsize=(10, 4.4))
vmax = np.percentile(np.abs(vf.v[0, :, :, k]), 99)
im0 = ax[0].imshow(vf.v[0, :, :, k], origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax[0].set_title(f"$v_x$ (z = {vf.redshift:.1f})"); fig.colorbar(im0, ax=ax[0])
dmax = np.percentile(np.abs(df.delta[:, :, k]), 99)
im1 = ax[1].imshow(df.delta[:, :, k], origin="lower", cmap="RdBu_r", vmin=-dmax, vmax=dmax)
ax[1].set_title(r"$\delta_1$ (icdensity)"); fig.colorbar(im1, ax=ax[1])
plt.tight_layout(); plt.show()

## Where to go next

**`subsample` is essential, not optional.** `Position` (float64) + `Velocity`
(float32) is 36 bytes/particle:

| `subsample` | particles (1536³ production) | Position+Velocity memory |
|---:|---:|---:|
| 1 | 3.62e9 | ~130 GB |
| 8 | 4.53e8 | ~16 GB |
| 64 (used above) | 5.66e7 | ~2.0 GB |
| 512 | 7.08e6 | ~0.25 GB |

(A full, unsubsampled load of the 512³ companion IC — smaller than production —
is already 1.34e8 particles, ~4.8 GB for `Position`+`Velocity`.) Once you need
the *full* box rather than a subsample, use `load_ic_velocity_mesh` /
`load_ic_density` instead of raw particles — they stream the bigfile in chunks
rather than holding every particle resident. `load_ic_density` never holds more
than one chunk plus the output mesh; `load_ic_velocity_mesh` is heavier — it
keeps a float64 momentum grid *per velocity component* (`3·nmesh³`, not
`nmesh³`), so its peak is **~4x** `load_ic_density`'s at the same `nmesh` (see
the README's IC memory table, and the note directly below it for the velocity
mesh's own numbers).

**Gas ≠ DM.** PRIYA runs GenIC with `ScaleDepVelocity`
(= `DifferentTransferFunctions`) `= 1`, so `ptype="gas"` is drawn from a
*different* CLASS transfer function than `ptype="dm"` — not a rescaled copy of
it. Load it the same way with `ptype="gas"` if your analysis needs the baryon
field specifically.

**Co-registering with τ.** The fields here are real-space, at z≈99; τ is
redshift-space, at the forest redshifts (z≈2.2–5.4). Growth-rescaling `δ₁` and
matching grids/orientation to the τ transverse plane is worked through in
[`HANDOFF.md`](../HANDOFF.md); the suite-wide walkthrough of parameters and τ is
in [`quickstart.ipynb`](quickstart.ipynb).